# <center>Exploratory Data Analysis (EDA) - Modern Pipeline</center>

## <center><i>Tabular Data - Python 3.12+ Compatible</i></center>

This notebook is a modernized version of the EDA pipeline, compatible with Python 3.12+. It uses:
- **ydata-profiling** instead of pandas-profiling
- **seaborn** and **matplotlib** instead of dataprep
- **sklearn.inspection** instead of rfpimp
- **sweetviz** for train/test comparison

All functionality is preserved with modern, maintained packages.

## Content

- [Setup & Data Loading](#setup)
- [I - EDA](#part_1)
    - [I-1 ydata-profiling](#part_1_1)
    - [I-2 Correlation Analysis](#part_1_2)
    - [I-3 Missing Value Analysis](#part_1_3)
    - [I-4 Train/Test Comparison with Sweetviz](#part_1_4)
    - [I-5 Target Identification](#part_1_5)
- [II - Feature Selection](#part_2)
    - [II-1 Removing Low Variance Features](#part_2_1)
    - [II-2 Univariate Selection](#part_2_2)
    - [II-3 Recursive Feature Elimination (RFE)](#part_2_3)
    - [II-4 SelectFromModel](#part_2_4)
- [III - Feature Extraction](#part_3)
    - [III-1 Principal Component Analysis (PCA)](#part_3_1)
    - [III-2 Independent Component Analysis (ICA)](#part_3_2)
    - [III-3 Linear Discriminant Analysis (LDA)](#part_3_3)
    - [III-4 Locally Linear Embedding (LLE)](#part_3_4)
    - [III-5 t-SNE](#part_3_5)
- [IV - Feature Importance](#part_4)
    - [IV-1 Tree-based Method](#part_4_1)
    - [IV-2 Permutation Importance (sklearn)](#part_4_2)

---
## <a id="setup">Setup & Data Loading</a>
---

In [ ]:
# Import core libraries
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Core libraries loaded")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

In [ ]:
# Import EDA tools
try:
    from ydata_profiling import ProfileReport
    print("✅ ydata-profiling loaded")
except ImportError:
    print("❌ ydata-profiling not found. Install with: pip install ydata-profiling")

try:
    import sweetviz
    print("✅ sweetviz loaded")
except ImportError:
    print("❌ sweetviz not found. Install with: pip install sweetviz")

In [ ]:
# Import ML libraries
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2, RFE, RFECV, SelectFromModel
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import locally_linear_embedding, TSNE
from sklearn.ensemble import ExtraTreesClassifier, RandomForestRegressor
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

print("✅ Scikit-learn modules loaded")

### Load Data

Choose one of the example datasets below, or load your own CSV file.

In [ ]:
# Option 1: Load sklearn California Housing dataset (regression)
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame
print(f"Loaded California Housing dataset: {df.shape}")
df.head()

In [ ]:
# Option 2: Load from URL
# df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
# print(f"Loaded dataset: {df.shape}")
# df.head()

In [ ]:
# Option 3: Load your own CSV
# df = pd.read_csv("your_file.csv")
# df.head()

---
## <a id="part_1">I - Exploratory Data Analysis</a>
---

### <a id="part_1_1">I-1 ydata-profiling (Automated EDA Report)</a>

In [ ]:
# Generate comprehensive EDA report
profile = ProfileReport(df, title='Automated EDA Report', explorative=True, minimal=False)
print("Report generated. Choose display method below:")

In [ ]:
# Display as interactive widgets
profile.to_widgets()

In [ ]:
# Or display as inline HTML
# profile.to_notebook_iframe()

In [ ]:
# Or save to HTML file
# profile.to_file("eda_report.html")
# print("Report saved to eda_report.html")

---
### <a id="part_1_2">I-2 Correlation Analysis</a>
---

In [ ]:
# Calculate correlation matrix
plt.figure(figsize=(12, 10))
correlation_matrix = df.select_dtypes(include=[np.number]).corr()

# Create heatmap
sns.heatmap(correlation_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8})

plt.title('Correlation Matrix Heatmap', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Show top correlations
def get_top_correlations(df, n=10):
    corr_matrix = df.select_dtypes(include=[np.number]).corr().abs()
    
    # Get upper triangle
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Find top correlations
    top_corr = upper.unstack().sort_values(ascending=False).head(n)
    
    return top_corr

print("Top 10 Feature Correlations:")
print(get_top_correlations(df, n=10))

---
### <a id="part_1_3">I-3 Missing Value Analysis</a>
---

In [ ]:
# Calculate missing values
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_data.index,
    'Missing_Count': missing_data.values,
    'Missing_Percent': missing_percent.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print("Missing Values Summary:")
    print(missing_df.to_string(index=False))
    
    # Visualize missing values
    plt.figure(figsize=(10, max(6, len(missing_df) * 0.5)))
    plt.barh(missing_df['Column'], missing_df['Missing_Percent'], color='coral')
    plt.xlabel('Missing Percentage (%)', fontsize=12)
    plt.title('Missing Values by Column', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values found in the dataset!")

In [ ]:
# Missing value heatmap
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Missing Value Heatmap (Yellow = Missing)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values to visualize.")

---
### <a id="part_1_4">I-4 Train/Test Comparison with Sweetviz</a>
---

In [ ]:
# Split data 80/20
train = df.iloc[:int(len(df)*0.8)]
test = df.iloc[int(len(df)*0.8):]

print(f"Train set: {train.shape}")
print(f"Test set: {test.shape}")

In [ ]:
# Generate comparison report (specify target column if you have one)
# Replace 'MedHouseVal' with your target column name
TARGET_COL = 'MedHouseVal' if 'MedHouseVal' in df.columns else df.columns[-1]

my_report = sweetviz.compare([train, "Train"], [test, "Test"], TARGET_COL)
my_report.show_html("train_test_comparison.html")
print("✅ Comparison report saved to train_test_comparison.html")

---
### <a id="part_1_5">I-5 Target Identification</a>
---

In [ ]:
# Display all columns
print("Available columns:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

In [ ]:
# Set your target variable
# Change this to your actual target column name
TARGET = 'MedHouseVal' if 'MedHouseVal' in df.columns else df.columns[-1]

print(f"Target variable: {TARGET}")

In [ ]:
# Separate features and target
y = df[TARGET].values
X = df.drop(columns=[TARGET])

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Detect problem type
continuous = False
if y.dtype in [np.float64, np.float32]:
    print("\n✅ Detected: REGRESSION problem (continuous target)")
    continuous = True
else:
    n_unique = len(np.unique(y))
    print(f"\n✅ Detected: CLASSIFICATION problem ({n_unique} classes)")
    print(f"Class distribution: {np.unique(y, return_counts=True)}")

In [ ]:
# Visualize target distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
if continuous:
    plt.hist(y, bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel(TARGET)
    plt.ylabel('Frequency')
    plt.title('Target Distribution (Histogram)')
else:
    unique, counts = np.unique(y, return_counts=True)
    plt.bar(unique, counts, edgecolor='black', alpha=0.7)
    plt.xlabel(TARGET)
    plt.ylabel('Count')
    plt.title('Target Distribution (Class Counts)')

plt.subplot(1, 2, 2)
if continuous:
    sns.boxplot(y=y)
    plt.ylabel(TARGET)
    plt.title('Target Distribution (Boxplot)')
else:
    unique, counts = np.unique(y, return_counts=True)
    plt.pie(counts, labels=unique, autopct='%1.1f%%')
    plt.title('Target Distribution (Pie Chart)')

plt.tight_layout()
plt.show()

---
## <a id="part_2">II - Feature Selection</a>
---

### <a id="part_2_1">II-1 Removing Features with Low Variance</a>

In [ ]:
# Remove features with low variance (>80% same value)
sel = VarianceThreshold(threshold=(.8 * (1 - .8)))
X_variance = sel.fit_transform(X)

print(f"Original number of features: {X.shape[1]}")
print(f"After variance threshold: {X_variance.shape[1]}")
print(f"Removed {X.shape[1] - X_variance.shape[1]} low-variance features")

---
### <a id="part_2_2">II-2 Univariate Selection (Chi-Square)</a>
---

**Note:** Chi-square test only works for classification problems.

In [ ]:
if not continuous:
    # Ensure all features are positive for chi2
    X_positive = X - X.min() + 1
    
    # Select top k features
    k = min(10, X.shape[1])  # Select top 10 or all features if less than 10
    select_best_features = SelectKBest(score_func=chi2, k=k)
    fit = select_best_features.fit(X_positive, y)
    
    # Create results dataframe
    df_scores = pd.DataFrame({
        'Feature': X.columns,
        'Score': fit.scores_
    }).sort_values('Score', ascending=False)
    
    print(f"Top {k} features by Chi-Square test:\n")
    print(df_scores.head(k).to_string(index=False))
    
    # Visualize
    plt.figure(figsize=(10, 6))
    plt.barh(df_scores.head(k)['Feature'], df_scores.head(k)['Score'])
    plt.xlabel('Chi-Square Score')
    plt.title(f'Top {k} Features by Univariate Selection')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping chi-square test (only for classification problems)")
    print("For regression, consider using f_regression or mutual_info_regression")

---
### <a id="part_2_3">II-3 Recursive Feature Elimination (RFE)</a>
---

In [ ]:
if not continuous:
    print("Running RFE with SVM (may take a few minutes for large datasets)...")
    
    # Sample data if too large
    if len(X) > 1000:
        sample_idx = np.random.choice(len(X), 1000, replace=False)
        X_sample = X.iloc[sample_idx]
        y_sample = y[sample_idx]
        print(f"Using sample of 1000 rows (original: {len(X)} rows)")
    else:
        X_sample = X
        y_sample = y
    
    svc = SVC(kernel="linear")
    rfecv = RFECV(estimator=svc, step=1, cv=StratifiedKFold(2), scoring='accuracy', n_jobs=-1)
    rfecv.fit(X_sample, y_sample)
    
    print(f"\nOptimal number of features: {rfecv.n_features_}")
    
    # Plot results
    plt.figure(figsize=(10, 6))
    plt.xlabel("Number of features selected")
    plt.ylabel("Cross-validation score")
    plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), 
             rfecv.cv_results_['mean_test_score'])
    plt.title('RFE with Cross-Validation')
    plt.tight_layout()
    plt.show()
    
    # Show selected features
    selected_features = X.columns[rfecv.support_]
    print(f"\nSelected features: {list(selected_features)}")
else:
    print("⚠️ Skipping RFE with SVC (for classification only)")
    print("For regression, you can use RFE with RandomForestRegressor or Ridge")

---
### <a id="part_2_4">II-4 SelectFromModel (L1-based)</a>
---

In [ ]:
# Use L1 regularization to select features
lsvc = LinearSVC(C=0.01, penalty="l1", dual=False, max_iter=5000)
lsvc.fit(X, y)

model = SelectFromModel(lsvc, prefit=True)
X_new = model.transform(X)

print(f"Original number of features: {X.shape[1]}")
print(f"Number of features selected: {X_new.shape[1]}")

# Show selected features
selected_features = X.columns[model.get_support()]
print(f"\nSelected features: {list(selected_features)}")

---
## <a id="part_3">III - Feature Extraction (Dimensionality Reduction)</a>
---

In [ ]:
# Set number of components for visualization
N_COMPONENTS = 2

### <a id="part_3_1">III-1 Principal Component Analysis (PCA)</a>

In [ ]:
# Apply PCA
pca = PCA(n_components=N_COMPONENTS)
X_pca = pca.fit_transform(X)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Visualize
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.6, edgecolors='k')
plt.colorbar(scatter, label=TARGET)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('PCA - First Two Principal Components')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PCA with more components to see cumulative variance
pca_full = PCA()
pca_full.fit(X)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
         np.cumsum(pca_full.explained_variance_ratio_), 
         marker='o', linestyle='--')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA - Cumulative Explained Variance')
plt.grid(True, alpha=0.3)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
plt.legend()
plt.tight_layout()
plt.show()

---
### <a id="part_3_2">III-2 Independent Component Analysis (ICA)</a>
---

In [ ]:
# Apply ICA
ica = FastICA(n_components=N_COMPONENTS, random_state=42)
X_ica = ica.fit_transform(X)

# Visualize
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_ica[:, 0], X_ica[:, 1], c=y, cmap='viridis', alpha=0.6, edgecolors='k')
plt.colorbar(scatter, label=TARGET)
plt.xlabel('IC1')
plt.ylabel('IC2')
plt.title('ICA - First Two Independent Components')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
### <a id="part_3_3">III-3 Linear Discriminant Analysis (LDA)</a>
---

**Note:** LDA only works for classification problems.

In [ ]:
if not continuous:
    # Apply LDA
    n_classes = len(np.unique(y))
    n_components_lda = min(N_COMPONENTS, n_classes - 1)
    
    lda = LinearDiscriminantAnalysis(n_components=n_components_lda)
    X_lda = lda.fit_transform(X, y)
    
    print(f"Original number of features: {X.shape[1]}")
    print(f"Reduced number of features: {X_lda.shape[1]}")
    print(f"Explained variance ratio: {lda.explained_variance_ratio_}")
    
    # Visualize
    if X_lda.shape[1] >= 2:
        plt.figure(figsize=(10, 6))
        scatter = plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='viridis', alpha=0.6, edgecolors='k')
        plt.colorbar(scatter, label=TARGET)
        plt.xlabel('LD1')
        plt.ylabel('LD2')
        plt.title('LDA - First Two Linear Discriminants')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        plt.figure(figsize=(10, 6))
        plt.hist(X_lda[:, 0], bins=30, alpha=0.6)
        plt.xlabel('LD1')
        plt.ylabel('Frequency')
        plt.title('LDA - First Linear Discriminant')
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ Skipping LDA (only for classification problems)")

---
### <a id="part_3_4">III-4 Locally Linear Embedding (LLE)</a>
---

In [ ]:
# Sample data if too large (LLE is computationally expensive)
if len(X) > 2000:
    sample_idx = np.random.choice(len(X), 2000, replace=False)
    X_sample = X.iloc[sample_idx].values
    y_sample = y[sample_idx]
    print(f"Using sample of 2000 rows (original: {len(X)} rows)")
else:
    X_sample = X.values
    y_sample = y

print("Applying LLE (may take a minute)...")
X_lle, error = locally_linear_embedding(X_sample, 
                                         n_neighbors=10, 
                                         n_components=N_COMPONENTS, 
                                         random_state=42, 
                                         n_jobs=-1)

print(f"Reconstruction error: {error:.4f}")

# Visualize
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_lle[:, 0], X_lle[:, 1], c=y_sample, cmap='viridis', alpha=0.6, edgecolors='k')
plt.colorbar(scatter, label=TARGET)
plt.xlabel('LLE1')
plt.ylabel('LLE2')
plt.title('Locally Linear Embedding')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
### <a id="part_3_5">III-5 t-distributed Stochastic Neighbor Embedding (t-SNE)</a>
---

In [ ]:
# Sample data if too large (t-SNE is very slow on large datasets)
if len(X) > 5000:
    sample_idx = np.random.choice(len(X), 5000, replace=False)
    X_sample = X.iloc[sample_idx].values
    y_sample = y[sample_idx]
    print(f"Using sample of 5000 rows (original: {len(X)} rows)")
else:
    X_sample = X.values
    y_sample = y

print("Applying t-SNE (this may take several minutes)...")
tsne = TSNE(n_components=N_COMPONENTS, random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(X_sample)

print("✅ t-SNE complete")

# Visualize
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='viridis', alpha=0.6, edgecolors='k')
plt.colorbar(scatter, label=TARGET)
plt.xlabel('t-SNE1')
plt.ylabel('t-SNE2')
plt.title('t-SNE Visualization')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## <a id="part_4">IV - Feature Importance</a>
---

### <a id="part_4_1">IV-1 Tree-based Feature Importance</a>

In [ ]:
if continuous:
    # Regression: Random Forest
    print("Training Random Forest Regressor...")
    rf = RandomForestRegressor(n_estimators=100, 
                               n_jobs=-1, 
                               oob_score=True, 
                               bootstrap=True, 
                               random_state=42)
    rf.fit(X, y)
    
    print(f"R² Training Score: {rf.score(X, y):.4f}")
    print(f"OOB Score: {rf.oob_score_:.4f}")
    
    # Get feature importances
    importances = rf.feature_importances_
    std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
    
else:
    # Classification: Extra Trees
    print("Training Extra Trees Classifier...")
    forest = ExtraTreesClassifier(n_estimators=250, random_state=42, n_jobs=-1)
    forest.fit(X, y)
    
    print(f"Training Score: {forest.score(X, y):.4f}")
    
    # Get feature importances
    importances = forest.feature_importances_
    std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)

# Create results dataframe
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances,
    'Std': std
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10).to_string(index=False))

In [ ]:
# Visualize feature importances
plt.figure(figsize=(10, max(6, len(feature_importance_df) * 0.3)))
plt.barh(feature_importance_df['Feature'], 
         feature_importance_df['Importance'], 
         xerr=feature_importance_df['Std'],
         color='skyblue',
         edgecolor='black')
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Tree-based Feature Importance', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

---
### <a id="part_4_2">IV-2 Permutation Importance (sklearn)</a>
---

**This is the modern, built-in alternative to rfpimp.**

In [ ]:
# Split data for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

In [ ]:
if continuous:
    # Use the Random Forest model from previous cell
    print("Retraining model on training set...")
    rf.fit(X_train, y_train)
    model_for_perm = rf
    scoring = 'r2'
else:
    # Use the Extra Trees model from previous cell  
    print("Retraining model on training set...")
    forest.fit(X_train, y_train)
    model_for_perm = forest
    scoring = 'accuracy'

print(f"Validation Score: {model_for_perm.score(X_val, y_val):.4f}")

In [ ]:
# Calculate permutation importance
print("Calculating permutation importance (this may take a few minutes)...")
perm_importance = permutation_importance(
    model_for_perm, 
    X_val, 
    y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring=scoring
)

print("✅ Permutation importance calculated")

In [ ]:
# Create results dataframe
perm_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_importance.importances_mean,
    'Std': perm_importance.importances_std
}).sort_values('Importance', ascending=False)

print("Permutation Feature Importance (Top 10):")
print(perm_df.head(10).to_string(index=False))

In [ ]:
# Visualize permutation importance
plt.figure(figsize=(10, max(6, len(perm_df) * 0.3)))
plt.barh(perm_df['Feature'], 
         perm_df['Importance'], 
         xerr=perm_df['Std'],
         color='lightcoral',
         edgecolor='black')
plt.xlabel('Permutation Importance', fontsize=12)
plt.title('Permutation Feature Importance (sklearn)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Compare tree-based vs permutation importance
comparison_df = pd.DataFrame({
    'Feature': X.columns,
    'Tree_Importance': feature_importance_df.set_index('Feature')['Importance'],
    'Perm_Importance': perm_df.set_index('Feature')['Importance']
}).sort_values('Perm_Importance', ascending=False)

print("\nComparison: Tree-based vs Permutation Importance (Top 10):")
print(comparison_df.head(10).to_string(index=False))

In [ ]:
# Side-by-side comparison plot
top_n = 10
top_features = comparison_df.head(top_n).index

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Tree-based importance
ax1.barh(range(top_n), 
         comparison_df.loc[top_features, 'Tree_Importance'],
         color='skyblue',
         edgecolor='black')
ax1.set_yticks(range(top_n))
ax1.set_yticklabels(top_features)
ax1.set_xlabel('Importance')
ax1.set_title('Tree-based Feature Importance')
ax1.invert_yaxis()

# Permutation importance
ax2.barh(range(top_n), 
         comparison_df.loc[top_features, 'Perm_Importance'],
         color='lightcoral',
         edgecolor='black')
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(top_features)
ax2.set_xlabel('Importance')
ax2.set_title('Permutation Feature Importance')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

---
## Summary

This notebook provides a complete EDA pipeline with:

1. **Automated EDA**: ydata-profiling for comprehensive reports
2. **Visual EDA**: Correlation matrices, missing value analysis
3. **Dataset Comparison**: Sweetviz for train/test comparison
4. **Feature Selection**: Variance threshold, univariate selection, RFE, L1-based
5. **Feature Extraction**: PCA, ICA, LDA, LLE, t-SNE
6. **Feature Importance**: Tree-based and permutation methods

All using Python 3.12+ compatible packages!

---

**Next Steps:**
- Experiment with different datasets
- Tune hyperparameters for feature selection
- Compare different dimensionality reduction techniques
- Use selected features to train machine learning models